# 🎨 Visionary AI — Colab Model Server

Run each cell top-to-bottom. At the end you will get a public `ngrok` URL — paste it into your local `.env` file.


In [ ]:
# ── Cell 1: Install dependencies ─────────────────────────────
!pip install diffusers transformers accelerate torch fastapi uvicorn pyngrok pillow -q
print('✅ Dependencies installed')

In [ ]:
# ── Cell 2: Authenticate with Hugging Face ───────────────────
from huggingface_hub import login
import os

# ⚠️  Replace with your own HF token (Settings → Access Tokens)
HF_TOKEN = "YOUR_HF_TOKEN_HERE"
login(token=HF_TOKEN)
os.environ['HF_TOKEN'] = HF_TOKEN
print('✅ Logged in to Hugging Face')

In [ ]:
# ── Cell 3: GPU check ────────────────────────────────────────
import torch, gc
print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU detected. Image/video generation will be very slow.')

In [ ]:
# ── Cell 4: Load models ──────────────────────────────────────
from diffusers import DiffusionPipeline
from diffusers.utils import export_to_video
from transformers import pipeline as hf_pipeline

MODELS_LOADED = {}

print('Loading Text-to-Image (Stable Diffusion 1.5)...')
t2i_pipe = DiffusionPipeline.from_pretrained(
    'stable-diffusion-v1-5/stable-diffusion-v1-5',
    torch_dtype=torch.float16
).to('cuda')
t2i_pipe.enable_attention_slicing()
MODELS_LOADED['text_to_image'] = True
print('✅ T2I loaded')

gc.collect(); torch.cuda.empty_cache()

print('Loading Text-to-Video (ModelScope 1.7B)...')
t2v_pipe = DiffusionPipeline.from_pretrained(
    'damo-vilab/text-to-video-ms-1.7b',
    torch_dtype=torch.float16
).to('cuda')
t2v_pipe.enable_attention_slicing()
MODELS_LOADED['text_to_video'] = True
print('✅ T2V loaded')

print('Loading LLM (Gemma 3 1B)...')
llm_pipe = hf_pipeline(
    'text-generation',
    model='google/gemma-3-1b-it',
    model_kwargs={'torch_dtype': torch.float16},
    device_map='auto'
)
MODELS_LOADED['qa_llm'] = True
print('✅ LLM loaded')

print(f'\n✅ All models ready: {list(MODELS_LOADED.keys())}')

In [ ]:
# ── Cell 5: Agent logic ──────────────────────────────────────
import base64, io, re
from PIL import Image

# Style modifier map — applied when style_hint is provided
STYLE_SUFFIXES = {
    'cinematic': ', cinematic lighting, 35mm film, anamorphic lens, shallow depth of field',
    'anime':     ', anime art style, vibrant colors, Studio Ghibli inspired, detailed',
    'photorealistic': ', photorealistic, 8K, DSLR, sharp focus, natural lighting',
    'watercolor': ', watercolor painting, soft edges, pastel tones, artistic',
    'pixel':     ', pixel art, 16-bit style, retro game aesthetic',
}

def image_to_base64(pil_image: Image.Image) -> str:
    buf = io.BytesIO()
    pil_image.save(buf, format='PNG')
    return base64.b64encode(buf.getvalue()).decode()

def video_to_base64(path: str) -> str:
    with open(path, 'rb') as f:
        return base64.b64encode(f.read()).decode()

def apply_style(prompt: str, style_hint: str | None) -> str:
    if style_hint and style_hint.lower() in STYLE_SUFFIXES:
        return prompt + STYLE_SUFFIXES[style_hint.lower()]
    return prompt

def expand_prompt(prompt: str, mode: str) -> str:
    """Use the LLM to enrich an image/video prompt with visual detail."""
    instruction = (
        f'You are a prompt engineer.\n'
        f'Rewrite this {mode} prompt with more visual detail, lighting, and atmosphere.\n'
        f'Return ONLY the improved prompt, no explanation.\n\n'
        f'Original: {prompt}\nImproved:'
    )
    try:
        out = llm_pipe(instruction, max_new_tokens=80, temperature=0.7,
                       do_sample=True, return_full_text=False)
        expanded = out[0]['generated_text'].strip().split('\n')[0].strip()
        if len(expanded) > 20:
            return expanded
    except Exception:
        pass
    return prompt

def classify_prompt(prompt: str) -> dict:
    """Keyword-based intent classifier: qa | image | video."""
    p = prompt.lower()

    video_kw = ['video', 'clip', 'animation', 'animate', 'footage', 'movie scene']
    image_kw = [
        'image', 'picture', 'photo', 'draw', 'paint', 'illustration',
        'generate a', 'create a', 'make a', 'show me a', 'sketch',
        'artwork', 'portrait', 'landscape', 'render', 'scenery', 'design'
    ]

    if any(k in p for k in video_kw):
        return {'type': 'video', 'expanded_prompt': expand_prompt(prompt, 'video')}
    if any(k in p for k in image_kw):
        return {'type': 'image', 'expanded_prompt': expand_prompt(prompt, 'image')}
    return {'type': 'qa'}

def run_agent(prompt: str, history: list = [],
              style_hint: str | None = None,
              num_frames: int = 8) -> dict:

    cls = classify_prompt(prompt)
    kind = cls['type']
    expanded = cls.get('expanded_prompt', prompt)
    final_prompt = apply_style(expanded, style_hint)

    print(f'[Agent] type={kind} | style={style_hint} | prompt={final_prompt[:80]}')

    if kind == 'image':
        result = t2i_pipe(final_prompt)
        return {
            'type': 'image',
            'prompt': final_prompt,
            'data': image_to_base64(result.images[0])
        }

    elif kind == 'video':
        frames_count = max(4, min(num_frames, 16))  # clamp 4–16
        result = t2v_pipe(final_prompt, num_frames=frames_count, num_inference_steps=50)
        video_path = export_to_video(result.frames[0], output_video_path='/tmp/output.mp4', fps=8)
        return {
            'type': 'video',
            'prompt': final_prompt,
            'frames': frames_count,
            'data': video_to_base64(video_path)
        }

    else:  # qa
        messages = [{'role': h['role'], 'content': h['content']} for h in history[-6:]]
        messages.append({'role': 'user', 'content': prompt})
        raw = llm_pipe(messages, max_new_tokens=1024, temperature=0.7, do_sample=True)
        generated = raw[0]['generated_text']
        answer = generated[-1].get('content', '') if isinstance(generated, list) else str(generated)
        return {'type': 'qa', 'prompt': prompt, 'data': answer}

print('✅ Agent functions ready')

In [ ]:
# ── Cell 6: Start FastAPI + expose via ngrok ─────────────────
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional
import uvicorn, threading
from pyngrok import ngrok

server_app = FastAPI(title='Visionary AI — Model Server')
server_app.add_middleware(CORSMiddleware, allow_origins=['*'],
                          allow_methods=['*'], allow_headers=['*'])

class PromptRequest(BaseModel):
    prompt: str
    history: list = []
    style_hint: Optional[str] = None
    num_frames: Optional[int] = 8

@server_app.get('/health')
def health():
    return {'status': 'ok', 'models_loaded': MODELS_LOADED}

@server_app.post('/generate')
def generate(req: PromptRequest):
    return run_agent(req.prompt, req.history, req.style_hint, req.num_frames or 8)

def run_server():
    uvicorn.run(server_app, host='0.0.0.0', port=8000)

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

# ⚠️  Replace with your own ngrok auth token (https://dashboard.ngrok.com/get-started/your-authtoken)
NGROK_AUTH_TOKEN = 'YOUR_NGROK_AUTH_TOKEN_HERE'
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_url = ngrok.connect(8000)
clean_url = str(public_url).replace('NgrokTunnel: "', '').split('"')[0]

print(f'\n{"="*55}')
print(f'  🚀 Visionary AI Model Server is LIVE')
print(f'  Public URL : {clean_url}')
print(f'  Health     : {clean_url}/health')
print(f'  Generate   : {clean_url}/generate  [POST]')
print(f'{"="*55}')
print(f'\nPaste into your .env file:')
print(f'COLAB_API_URL={clean_url}')